In [ ]:
import pandas as pd

df = pd.read_csv(r"/home/ayoub/Desktop/darkom_annonces/data/darkom_annonces.csv")

df.info()

In [ ]:
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

df.info()

In [ ]:
df.head()

In [ ]:
df["date_publication"] = pd.to_datetime(df["date_publication"], format="%Y-%m-%d", errors="coerce")

df[["type_bien", "transaction"]] = df[
    ["type_bien", "transaction"]
].astype("category")

df["annee_construction"] = df["annee_construction"].astype("Int64")

In [ ]:
numeric_columns = [
    "prix",
    "nb_chambres",
    "nb_salles_bain",
    "etage",
    "annee_construction"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).round().astype("Int64")

df.info()

In [ ]:
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

df = df.sort_values("date_publication").reset_index(drop=True)

df["date_publication"] = (
    df["date_publication"]
    .ffill()
    .bfill()
)

In [ ]:
df["quartier"] = df.groupby("ville")["quartier"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

In [ ]:
df.info()

In [ ]:
df["annee_construction"].unique()

In [ ]:
df["type_bien"].unique()

In [ ]:
def extract_type_bien(title):

    title = title.lower()

    if "appartement" in title:
        return "Appartement"

    elif "villa" in title:
        return "Villa"

    elif "bureau" in title:
        return "Bureau"

    elif "terrain" in title:
        return "Terrain"

    elif "duplex" in title:
        return "Duplex"

    else:
        return None

df["type_bien"] = df["type_bien"].fillna(
    df["titre"].apply(extract_type_bien)
)


In [ ]:
# =============================
# STEP 1 : Basic text cleaning
# =============================


# Clean ville column
ville_mapping = {
    "casablanca": "Casablanca",
    "casa": "Casablanca",

    "rabat": "Rabat",

    "marrakech": "Marrakech",

    "fès": "Fes",
    "fes": "Fes",

    "tanger": "Tanger",

    "agadir": "Agadir",

    "meknès": "Meknes",
    "meknes": "Meknes",

    "oujda": "Oujda",

    "kenitra": "Kenitra",

    "tétouan": "Tetouan",
    "tetouan": "Tetouan",
}

df["ville"] = (
    df["ville"]
    .str.strip()
    .str.lower()
    .replace(ville_mapping)
)

 
# ========================
# STEP 2 : FIX DATA TYPES
# ========================


# Convert date_publication column to datetime
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

# Convert numeric columns from float to integer
numeric_columns = [
    "prix",
    "nb_chambres",
    "nb_salles_bain",
    "etage",
    "annee_construction"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).round().astype("Int64")

# Convert string columns to category dtype
df[["type_bien", "transaction"]] = (
    df[["type_bien", "transaction"]]
    .astype("category")
)



# ===============================
# STEP 3 : HANDLE MISSING VALUES
# ===============================


# Fix missing values in date_publication column (5% missing values)
df = df.sort_values("date_publication").reset_index(drop=True)

df["date_publication"] = (
    df["date_publication"]
    .ffill()
    .bfill()
)

# Fix missing values in quartier column (27% missing values)
df["quartier"] = df.groupby("ville")["quartier"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

# Fix missing values in type_bien column (2.5% missing values)

# Extract type_bien from titre
def extract_type_bien(title):

    title = title.lower()

    if "appartement" in title:
        return "Appartement"

    elif "villa" in title:
        return "Villa"

    elif "bureau" in title:
        return "Bureau"

    elif "terrain" in title:
        return "Terrain"

    elif "duplex" in title:
        return "Duplex"

    else:
        return None

df["type_bien"] = df["type_bien"].fillna(
    df["titre"].apply(extract_type_bien)
)

# Fix missing values in transaction column (2.5% missing values)
def fill_transaction(row):

    if pd.notna(row["transaction"]):
        return row["transaction"]

    if row["prix"] <= 30000:
        return "Location"

    else:
        return "Vente"

df["transaction"] = df.apply(
    fill_transaction,
    axis=1
)

# Fix missing values in nb_chambres (8.5%), nb_salles_bain (6.9%), and etage (15.4%)
columns_to_fill = [
    "nb_chambres",
    "nb_salles_bain",
    "etage"
]

for col in columns_to_fill:

    df.loc[
        df["type_bien"] == "Terrain",
        col
    ] = df.loc[
        df["type_bien"] == "Terrain",
        col
    ].fillna(0)

for col in columns_to_fill:

    df[col] = df.groupby("type_bien")[col].transform(
        lambda x: x.fillna(x.median())
    )

In [ ]:
df["annee_construction"] = df.groupby(
    ["ville", "type_bien"]
)["annee_construction"].transform(
    lambda x: x.fillna(round(x.median()))
)

df.info()

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.groupby("transaction")["prix"].describe()

In [ ]:
df.groupby("transaction")["prix"].quantile([0.90, 0.95, 0.99])

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("darkom-annonces-6a0a532a16460470060059.csv")

# ════════════════════════════════════════════════════════════════
# PARTIE 1 — NETTOYAGE DE BASE (étapes précédentes)
# ════════════════════════════════════════════════════════════════

# STEP 1 — Drop duplicates sur annonce_id
df = df.drop_duplicates(subset="annonce_id", keep="first")

# STEP 2 — Sort par date_publication + ffill/bfill
df["date_publication"] = pd.to_datetime(df["date_publication"])
df = df.sort_values("date_publication").reset_index(drop=True)
df["date_publication"] = df["date_publication"].ffill().bfill()

# STEP 3 — Remplir quartier avec le mode par ville
def fill_quartier(group):
    mode_vals = group["quartier"].mode()
    if not mode_vals.empty:
        group["quartier"] = group["quartier"].fillna(mode_vals[0])
    return group

df = df.groupby("ville", group_keys=False).apply(fill_quartier)

# STEP 4 — Extraire type_bien depuis les 3 premiers mots du titre
def extract_type_from_titre(row):
    if pd.notna(row["type_bien"]):
        return row["type_bien"]
    titre = str(row["titre"]).lower()
    first_words = " ".join(titre.split()[:3])
    if "appartement" in first_words:
        return "Appartement"
    elif "villa" in first_words:
        return "Villa"
    elif "terrain" in first_words:
        return "Terrain"
    elif "bureau" in first_words:
        return "Bureau"
    elif "duplex" in first_words:
        return "Duplex"
    return row["type_bien"]

df["type_bien"] = df.apply(extract_type_from_titre, axis=1)

# STEP 5 — Remplir transaction selon intervalle de prix
SEUIL = 50_000

def fill_transaction(row):
    if pd.notna(row["transaction"]):
        return row["transaction"]
    return "Vente" if row["prix"] >= SEUIL else "Location"

df["transaction"] = df.apply(fill_transaction, axis=1)

# STEP 6 — Remplir nb_chambres, nb_salles_bain, etage avec médiane par type_bien
for col in ["nb_chambres", "nb_salles_bain", "etage"]:
    medians = df.groupby("type_bien")[col].transform("median")
    df[col] = df[col].fillna(medians)


# ════════════════════════════════════════════════════════════════
# PARTIE 2 — DÉTECTION ET SUPPRESSION DES VALEURS ABERRANTES
# Méthode IQR : on supprime les lignes hors [Q1-1.5*IQR, Q3+1.5*IQR]
# ════════════════════════════════════════════════════════════════

def iqr_mask(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series >= lower) & (series <= upper)

before = len(df)

df = df[
    iqr_mask(df["prix"])        &
    iqr_mask(df["surface"])     &
    iqr_mask(df["nb_chambres"])
].reset_index(drop=True)

print(f"🗑️  Outliers supprimés : {before - len(df)} lignes  |  Restantes : {len(df)}")


# ════════════════════════════════════════════════════════════════
# PARTIE 3 — STANDARDISATION DES DONNÉES
# ════════════════════════════════════════════════════════════════

# 3a — Uniformiser les noms de villes
ville_mapping = {
    "casablanca"  : "Casablanca",
    "casa"        : "Casablanca",
    "rabat"       : "Rabat",
    "marrakech"   : "Marrakech",
    "marrakesh"   : "Marrakech",
    "fes"         : "Fès",
    "fez"         : "Fès",
    "tanger"      : "Tanger",
    "tangier"     : "Tanger",
    "agadir"      : "Agadir",
    "meknes"      : "Meknès",
    "oujda"       : "Oujda",
    "kenitra"     : "Kénitra",
    "tetouan"     : "Tétouan",
    "sale"        : "Salé",
    "temara"      : "Témara",
    "mohammedia"  : "Mohammedia",
    "el jadida"   : "El Jadida",
    "beni mellal" : "Beni Mellal",
    "nador"       : "Nador",
}

df["ville"] = (
    df["ville"]
    .str.strip()
    .str.lower()
    .replace(ville_mapping)
    .str.title()
)

# 3b — Harmoniser les types de bien
type_mapping = {
    "appartement" : "Appartement",
    "app"         : "Appartement",
    "appart"      : "Appartement",
    "villa"       : "Villa",
    "terrain"     : "Terrain",
    "bureau"      : "Bureau",
    "duplex"      : "Duplex",
}

df["type_bien"] = (
    df["type_bien"]
    .str.strip()
    .str.lower()
    .replace(type_mapping)
    .str.capitalize()
)

# 3c — Normaliser les transactions
transaction_mapping = {
    "vente"    : "Vente",
    "sale"     : "Vente",
    "location" : "Location",
    "louer"    : "Location",
    "rent"     : "Location",
}

df["transaction"] = (
    df["transaction"]
    .str.strip()
    .str.lower()
    .replace(transaction_mapping)
    .str.capitalize()
)


# ════════════════════════════════════════════════════════════════
# PARTIE 4 — CORRECTION DES TYPES DE DONNÉES
# ════════════════════════════════════════════════════════════════

# date_publication déjà en datetime depuis STEP 2
df["date_publication"] = pd.to_datetime(df["date_publication"])

# Types numériques
numeric_cols = ["prix", "surface", "nb_chambres", "nb_salles_bain", "etage", "annee_construction"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Entiers là où ça fait sens (Int64 supporte les NaN)
for col in ["nb_chambres", "nb_salles_bain", "etage", "annee_construction"]:
    df[col] = df[col].round(0).astype("Int64")


# ════════════════════════════════════════════════════════════════
# PARTIE 5 — FEATURE ENGINEERING
# ════════════════════════════════════════════════════════════════

ANNEE_COURANTE = 2024

# 5a — Prix par m²
df["prix_par_m2"] = (df["prix"] / df["surface"]).round(2)

# 5b — Âge estimé du bien
df["age_bien"] = (ANNEE_COURANTE - df["annee_construction"]).astype("Int64")

# 5c — Catégorie de prix (seuils différents Vente vs Location)
def categoriser_prix(row):
    prix = row["prix"]
    if pd.isna(prix):
        return np.nan
    if row["transaction"] == "Vente":
        if prix < 500_000:
            return "Économique"
        elif prix < 1_500_000:
            return "Moyen"
        elif prix < 4_000_000:
            return "Haut standing"
        else:
            return "Luxe"
    else:  # Location
        if prix < 3_000:
            return "Économique"
        elif prix < 8_000:
            return "Moyen"
        elif prix < 20_000:
            return "Haut standing"
        else:
            return "Luxe"

df["categorie_prix"] = df.apply(categoriser_prix, axis=1)

# 5d — Catégorie de surface
def categoriser_surface(surface):
    if pd.isna(surface):
        return np.nan
    if surface < 80:
        return "Petit"
    elif surface <= 150:
        return "Moyen"
    else:
        return "Grand"

df["categorie_surface"] = df["surface"].apply(categoriser_surface)

# 5e — Dimensions temporelles
df["annee_publication"]     = df["date_publication"].dt.year
df["mois_publication"]      = df["date_publication"].dt.month
df["trimestre_publication"] = df["date_publication"].dt.quarter


# ════════════════════════════════════════════════════════════════
# SAVE
# ════════════════════════════════════════════════════════════════
df.to_csv("darkom_cleaned.csv", index=False)

print("✅ Pipeline complet — shape:", df.shape)
print("\n📋 Types des colonnes finales:")
print(df.dtypes)
print("\n🔍 Valeurs manquantes restantes:")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  Aucune valeur manquante !")
print("\n📊 Aperçu des nouvelles features:")
print(df[["prix", "surface", "prix_par_m2", "age_bien",
          "categorie_prix", "categorie_surface",
          "annee_publication", "mois_publication", "trimestre_publication"]].head(5).to_string())

In [105]:
import os
import logging
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv



# Database connection
load_dotenv()

def get_engine():
    try:
        engine = create_engine(
            f"postgresql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
            f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
        )

        logging.info("Database connection created successfully")
        return engine

    except Exception as e:
        logging.error(f"Error creating database connection: {e}")
        raise
engine = get_engine()

query = "SELECT * FROM staging.darkom_annonces"

df = pd.read_sql(query, engine)

# =============================
# STEP 1 : Basic text cleaning
# =============================

logging.info("STEP 1 START - Basic text cleaning")

# Clean ville column
ville_mapping = {
    "casablanca": "Casablanca",
    "casa": "Casablanca",

    "rabat": "Rabat",

    "marrakech": "Marrakech",

    "fès": "Fes",
    "fes": "Fes",

    "tanger": "Tanger",

    "agadir": "Agadir",

    "meknès": "Meknes",
    "meknes": "Meknes",

    "oujda": "Oujda",

    "kenitra": "Kenitra",

    "tétouan": "Tetouan",
    "tetouan": "Tetouan",
}

df["ville"] = (
    df["ville"]
    .str.strip()
    .str.lower()
    .replace(ville_mapping)
)

logging.info("STEP 1 END - Basic cleaning completed")
 
# ========================
# STEP 2 : FIX DATA TYPES
# ========================

logging.info("STEP 2 START - Converting data types")

# Convert date_publication column to datetime
df["date_publication"] = pd.to_datetime(
    df["date_publication"],
    errors="coerce"
)

# Convert numeric columns from float to integer
numeric_columns = [
    "prix",
    "nb_chambres",
    "nb_salles_bain",
    "etage",
    "annee_construction"
]

for col in numeric_columns:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    ).round().astype("Int64")

# Convert string columns to category dtype
df[["type_bien", "transaction"]] = (
    df[["type_bien", "transaction"]]
    .astype("category")
)

logging.info("STEP 2 END - Data types converted")


# ===============================
# STEP 3 : HANDLE MISSING VALUES
# ===============================

logging.info("STEP 3 START - Handling missing values")

# Fix missing values in date_publication column (5% missing values)
df = df.sort_values("date_publication").reset_index(drop=True)

df["date_publication"] = (
    df["date_publication"]
    .ffill()
    .bfill()
)

# Fix missing values in quartier column (27% missing values)
df["quartier"] = df.groupby("ville")["quartier"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else "Unknown")
)

# Fix missing values in type_bien column (2.5% missing values)

# Extract type_bien from titre
def extract_type_bien(title):

    title = title.lower()

    if "appartement" in title:
        return "Appartement"

    elif "villa" in title:
        return "Villa"

    elif "bureau" in title:
        return "Bureau"

    elif "terrain" in title:
        return "Terrain"

    elif "duplex" in title:
        return "Duplex"

    else:
        return None

df["type_bien"] = df["type_bien"].fillna(
    df["titre"].apply(extract_type_bien)
)

# Fix missing values in transaction column (2.5% missing values)
def fill_transaction(row):

    if pd.notna(row["transaction"]):
        return row["transaction"]

    if row["prix"] <= 30000:
        return "Location"

    else:
        return "Vente"

df["transaction"] = df.apply(
    fill_transaction,
    axis=1
)

# Fix missing values in nb_chambres (8.5%), nb_salles_bain (6.9%), and etage (15.4%)
columns_to_fill = [
    "nb_chambres",
    "nb_salles_bain",
    "etage"
]

for col in columns_to_fill:

    df.loc[
        df["type_bien"] == "Terrain",
        col
    ] = df.loc[
        df["type_bien"] == "Terrain",
        col
    ].fillna(0)

for col in columns_to_fill:

    df[col] = df.groupby("type_bien")[col].transform(
        lambda x: x.fillna(x.median())
    )

# Fix missing values in anne_construction (13.5% missing values)
df["annee_construction"] = df.groupby(
    ["ville", "type_bien"]
)["annee_construction"].transform(
    lambda x: x.fillna(round(x.median()))
)

logging.info("STEP 3 END - Missing values handled")

df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   annonce_id          1500 non-null   str           
 1   date_publication    1500 non-null   datetime64[us]
 2   titre               1500 non-null   str           
 3   ville               1500 non-null   str           
 4   quartier            1500 non-null   str           
 5   type_bien           1500 non-null   category      
 6   transaction         1500 non-null   str           
 7   prix                1500 non-null   Int64         
 8   surface             1500 non-null   float64       
 9   nb_chambres         1500 non-null   Int64         
 10  nb_salles_bain      1500 non-null   Int64         
 11  etage               1500 non-null   Int64         
 12  annee_construction  1500 non-null   Int64         
dtypes: Int64(5), category(1), datetime64[us](1), float64(1), st

In [ ]:
# Function to calculate outlier percentage using IQR

def outlier_percentage(df, column):

    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    outlier_count = len(outliers)

    total_count = len(df)

    percentage = (
        outlier_count / total_count
    ) * 100

    print(f"\nColumn: {column}")

    print(f"Outliers: {outlier_count}")

    print(f"Percentage: {percentage:.2f}%")

In [ ]:
outlier_percentage(df, "prix")

outlier_percentage(df, "surface")

outlier_percentage(df, "nb_chambres")

outlier_percentage(df, "nb_salles_bain")

In [ ]:
# ==================================================
# Detect and display outliers using IQR
# ==================================================

def show_outliers(df, column):

    # Calculate quartiles
    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Define bounds
    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    # Filter outliers
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    # Calculate statistics
    outlier_count = len(outliers)

    percentage = (
        outlier_count / len(df)
    ) * 100

    # Display information
    print("=" * 50)

    print(f"Column: {column}")

    print(f"Lower bound: {lower_bound}")

    print(f"Upper bound: {upper_bound}")

    print(f"Outliers count: {outlier_count}")

    print(f"Outliers percentage: {percentage:.2f}%")

    print("=" * 50)

    # Return important columns
    return outliers[
        [
            "titre",
            "ville",
            "type_bien",
            "transaction",
            "prix",
            "surface",
            "nb_chambres",
            "nb_salles_bain"
        ]
    ]


# ==================================================
# Prix outliers
# ==================================================

prix_outliers = show_outliers(df, "prix")

display(prix_outliers)


# ==================================================
# Surface outliers
# ==================================================

surface_outliers = show_outliers(df, "surface")

display(surface_outliers)


# ==================================================
# Chambres outliers
# ==================================================

chambres_outliers = show_outliers(df, "nb_chambres")

display(chambres_outliers)


# ==================================================
# Salle bain outliers
# ==================================================

sdb_outliers = show_outliers(df, "nb_salles_bain")

display(sdb_outliers)

In [ ]:
df = df[df["nb_chambres"] <= 10]

df = df[df["nb_salles_bain"] <= 6]

df = df[df["prix"] <= 20000000]

In [ ]:
df.info()

In [112]:
# ==================================================
# Detect and display outliers using IQR
# ==================================================

def show_outliers(df, column):

    # Calculate quartiles
    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Define bounds
    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    # Filter outliers
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    # Calculate statistics
    outlier_count = len(outliers)

    percentage = (
        outlier_count / len(df)
    ) * 100

    # Display information
    print("=" * 50)

    print(f"Column: {column}")

    print(f"Lower bound: {lower_bound}")

    print(f"Upper bound: {upper_bound}")

    print(f"Outliers count: {outlier_count}")

    print(f"Outliers percentage: {percentage:.2f}%")

    print("=" * 50)

    # Return important columns
    return outliers[
        [
            "titre",
            "ville",
            "type_bien",
            "transaction",
            "prix",
            "surface",
            "nb_chambres",
            "nb_salles_bain"
        ]
    ]


# ==================================================
# Prix outliers
# ==================================================

prix_outliers = show_outliers(df, "prix")

display(prix_outliers)


# ==================================================
# Surface outliers
# ==================================================

surface_outliers = show_outliers(df, "surface")

display(surface_outliers)


# ==================================================
# Chambres outliers
# ==================================================

chambres_outliers = show_outliers(df, "nb_chambres")

display(chambres_outliers)


# ==================================================
# Salle bain outliers
# ==================================================

sdb_outliers = show_outliers(df, "nb_salles_bain")

display(sdb_outliers)

Column: prix
Lower bound: -2666423.5
Upper bound: 4474702.5
Outliers count: 90
Outliers percentage: 6.00%


,titre,ville,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain
6,Villa moderne Oujda,Oujda,Villa,Vente,6405685,245.98,4,2
8,Beau Villa Casablanca,Casablanca,Villa,Vente,5592517,383.88,5,2
12,Beau Villa Oujda,Oujda,Villa,Vente,8324902,187.34,6,4
14,Villa 326m² Oujda,Oujda,Villa,Vente,8176715,326.74,4,2
17,Villa moderne Oujda,Oujda,Villa,Vente,5721231,274.80,6,4
...,...,...,...,...,...,...,...,...
1449,Beau Villa Casablanca,Casablanca,Villa,Vente,7322460,249.36,3,1
1457,Villa à vente - Marrakech,Marrakech,Villa,Vente,4590145,316.36,5,2
1467,Villa moderne Oujda,Oujda,Villa,Vente,6295974,200.59,4,2
1486,Villa moderne Casablanca,Casablanca,Villa,Vente,6391011,205.98,5,3


Column: surface
Lower bound: -126.24374999999998
Upper bound: 423.42625
Outliers count: 40
Outliers percentage: 2.67%


,titre,ville,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain
23,Terrain moderne Tanger,Tanger,Terrain,Vente,185018,439.09,0,0
26,Terrain à vente - Marrakech,Marrakech,Terrain,Vente,345277,485.70,0,0
68,Terrain 491m² Centre,Kenitra,Terrain,Vente,722361,491.84,0,0
122,Terrain moderne Rabat,Rabat,Terrain,Vente,561192,429.12,0,0
141,Terrain moderne Kenitra,Kenitra,Terrain,Location,3842,447.48,0,0
251,Beau Appartement Fès,Fes,Appartement,Location,7646,2500.00,3,1
271,Terrain moderne Fès,Fes,Terrain,Vente,1089619,448.34,0,0
324,Terrain à vente - Agadir,Agadir,Terrain,Vente,971267,436.27,0,0
362,Villa à vente - Meknès,Meknes,Villa,Vente,7782590,452.15,5,2
512,Terrain moderne Oujda,Oujda,Terrain,Vente,913930,2500.00,0,0


Column: nb_chambres
Lower bound: -3.5
Upper bound: 8.5
Outliers count: 0
Outliers percentage: 0.00%


,titre,ville,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain


Column: nb_salles_bain
Lower bound: -0.5
Upper bound: 3.5
Outliers count: 21
Outliers percentage: 1.40%


,titre,ville,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain
0,Villa moderne Agadir,Agadir,Villa,Vente,2592302,248.46,6,4
12,Beau Villa Oujda,Oujda,Villa,Vente,8324902,187.34,6,4
17,Villa moderne Oujda,Oujda,Villa,Vente,5721231,274.80,6,4
151,Villa 286m² Zouagha,Fes,Villa,Vente,2183194,286.38,6,4
162,Villa moderne Kenitra,Kenitra,Villa,Vente,6336352,222.29,6,4
294,Villa 204m² Ville Nouvelle,Meknes,Villa,Vente,2752950,204.88,6,4
295,Villa 263m² Tétouan,Tetouan,Villa,Location,18476,263.55,6,4
380,Villa à vente - Meknès,Meknes,Villa,Vente,2453918,293.06,6,4
400,Villa moderne Agadir,Agadir,Villa,Location,2919,302.47,6,4
414,Villa à vente - Agadir,Agadir,Villa,Vente,1892838,193.22,6,4


In [ ]:
df["prix"].sort_values(ascending=False).head(20)

In [ ]:
df.info()

In [106]:
# Replace unrealistic nb_chambres values

mask = df["nb_chambres"] > 10

df.loc[mask, "nb_chambres"] = df.groupby(
    ["ville", "type_bien"]
)["nb_chambres"].transform("median")

In [107]:
# Replace unrealistic nb_salles_bain values

mask = df["nb_salles_bain"] > 6

df.loc[mask, "nb_salles_bain"] = df.groupby(
    ["ville", "type_bien"]
)["nb_salles_bain"].transform("median")

In [110]:
mask = df["prix"] > 20000000

median_prices = df.groupby(
    ["ville", "type_bien"]
)["prix"].transform("median").round()

df.loc[mask, "prix"] = median_prices[mask]

df["prix"] = df["prix"].astype("Int64")

In [111]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   annonce_id          1500 non-null   str           
 1   date_publication    1500 non-null   datetime64[us]
 2   titre               1500 non-null   str           
 3   ville               1500 non-null   str           
 4   quartier            1500 non-null   str           
 5   type_bien           1500 non-null   category      
 6   transaction         1500 non-null   str           
 7   prix                1500 non-null   Int64         
 8   surface             1500 non-null   float64       
 9   nb_chambres         1500 non-null   Int64         
 10  nb_salles_bain      1500 non-null   Int64         
 11  etage               1500 non-null   Int64         
 12  annee_construction  1500 non-null   Int64         
dtypes: Int64(5), category(1), datetime64[us](1), float64(1), st

In [113]:
df.head(30)

,annonce_id,date_publication,titre,ville,quartier,type_bien,transaction,prix,surface,nb_chambres,nb_salles_bain,etage,annee_construction
0,ANO000708,2023-01-01,Villa moderne Agadir,Agadir,Talborjt,Villa,Vente,2592302,248.46,6,4,0,2007
1,ANO001039,2023-01-01,Appartement à vente - Kenitra,Kenitra,Hay Essalam,Appartement,Vente,2155591,91.29,2,1,2,2008
2,ANO000870,2023-01-02,Bureau à vente - Tétouan,Tetouan,M'diq,Bureau,Vente,317774,155.88,0,0,1,2020
3,ANO000036,2023-01-03,Appartement 132m² Meknès,Meknes,Hamria,Appartement,Vente,598486,132.15,1,1,2,2011
4,ANO000883,2023-01-03,Terrain à location - Meknès,Meknes,Ville Nouvelle,Terrain,Location,2755,12.00,0,0,0,2009
5,ANO001275,2023-01-03,Beau Villa Meknès,Meknes,Hamria,Villa,Vente,2177553,264.60,4,2,0,2004
6,ANO000171,2023-01-05,Villa moderne Oujda,Oujda,Centre,Villa,Vente,6405685,245.98,4,2,0,2008
7,ANO001170,2023-01-05,Villa 290m² Marrakech,Marrakech,Medina,Villa,Vente,2169498,290.02,4,3,0,2005
8,ANO000802,2023-01-07,Beau Villa Casablanca,Casablanca,Hay Mohammadi,Villa,Vente,5592517,383.88,5,2,0,2020
9,ANO000385,2023-01-07,Villa 144m² Ville Nouvelle,Fes,Ville Nouvelle,Villa,Vente,1880519,144.15,4,2,0,2024


In [114]:
# ==================================================
# STEP 4 : DETECT & REMOVE OUTLIERS
# ==================================================

logging.info("STEP 4 START - Detecting outliers")


# --------------------------------------------------
# Function to detect outliers using IQR
# --------------------------------------------------

def detect_outliers(df, column):

    # Calculate quartiles
    Q1 = df[column].quantile(0.25)

    Q3 = df[column].quantile(0.75)

    # Calculate IQR
    IQR = Q3 - Q1

    # Define bounds
    lower_bound = Q1 - (1.5 * IQR)

    upper_bound = Q3 + (1.5 * IQR)

    # Detect outliers
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    # Calculate percentage
    percentage = (
        len(outliers) / len(df)
    ) * 100

    # Print results
    print("=" * 50)

    print(f"Column: {column}")

    print(f"Outliers count: {len(outliers)}")

    print(f"Outliers percentage: {percentage:.2f}%")

    print("=" * 50)

    return outliers


# --------------------------------------------------
# Detect outliers
# --------------------------------------------------

prix_outliers = detect_outliers(df, "prix")

surface_outliers = detect_outliers(df, "surface")

chambres_outliers = detect_outliers(df, "nb_chambres")

sdb_outliers = detect_outliers(df, "nb_salles_bain")


# --------------------------------------------------
# Calculate total outliers percentage
# --------------------------------------------------

total_outliers = (
    len(prix_outliers) +
    len(surface_outliers) +
    len(chambres_outliers) +
    len(sdb_outliers)
)

total_percentage = (
    total_outliers / len(df)
) * 100

print("\n" + "=" * 50)

print(f"Total outliers: {total_outliers}")

print(f"Total outliers percentage: {total_percentage:.2f}%")

print("=" * 50)


# --------------------------------------------------
# Remove unrealistic outliers
# --------------------------------------------------

# Remove unrealistic room values
df = df[df["nb_chambres"] <= 10]

df = df[df["nb_salles_bain"] <= 6]

# Remove extreme price values
df = df[df["prix"] <= 20000000]


logging.info("STEP 4 END - Outliers handled successfully")

Column: prix
Outliers count: 90
Outliers percentage: 6.00%
Column: surface
Outliers count: 40
Outliers percentage: 2.67%
Column: nb_chambres
Outliers count: 0
Outliers percentage: 0.00%
Column: nb_salles_bain
Outliers count: 21
Outliers percentage: 1.40%

Total outliers: 151
Total outliers percentage: 10.07%


In [115]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   annonce_id          1500 non-null   str           
 1   date_publication    1500 non-null   datetime64[us]
 2   titre               1500 non-null   str           
 3   ville               1500 non-null   str           
 4   quartier            1500 non-null   str           
 5   type_bien           1500 non-null   category      
 6   transaction         1500 non-null   str           
 7   prix                1500 non-null   Int64         
 8   surface             1500 non-null   float64       
 9   nb_chambres         1500 non-null   Int64         
 10  nb_salles_bain      1500 non-null   Int64         
 11  etage               1500 non-null   Int64         
 12  annee_construction  1500 non-null   Int64         
dtypes: Int64(5), category(1), datetime64[us](1), float64(1), st

In [116]:
dl = pd.read_csv(r"/home/ayoub/Desktop/darkom_annonces/data/cleaned_data.csv")

dl.info()

<class 'pandas.DataFrame'>
RangeIndex: 1472 entries, 0 to 1471
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   annonce_id          1472 non-null   str    
 1   date_publication    1472 non-null   str    
 2   titre               1472 non-null   str    
 3   ville               1472 non-null   str    
 4   quartier            1472 non-null   str    
 5   type_bien           1472 non-null   str    
 6   transaction         1472 non-null   str    
 7   prix                1472 non-null   int64  
 8   surface             1472 non-null   float64
 9   nb_chambres         1472 non-null   int64  
 10  nb_salles_bain      1472 non-null   int64  
 11  etage               1472 non-null   int64  
 12  annee_construction  1472 non-null   int64  
dtypes: float64(1), int64(5), str(7)
memory usage: 149.6 KB
